# Semana 20 · ¿Dónde está el punto en torno al que giran dos cuerpos?

Este cuaderno acompaña a la sesión de proyecto de la semana 20: el centro de masas y la simulación de dos cuerpos que se atraen.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Centros de masas

Con NumPy, el centro de masas de varios cuerpos es una sola línea: se multiplica cada posición por su masa, se suman y se divide por la masa total. `np.average` con el argumento `weights` hace exactamente una media ponderada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

masas = np.array([2.0, 6.0])
posiciones = np.array([[0.0, 0.0],
                       [4.0, 4.0]])     # una fila por cuerpo
cm = np.average(posiciones, axis=0, weights=masas)
print("Centro de masas:", cm)

# Tierra y Luna sobre una recta, en km
m_T, m_L, d = 81.3, 1.0, 384_400
print(f"Centro de masas Tierra-Luna: {d * m_L / (m_T + m_L):.0f} km del centro de la Tierra")

## 2. Unidades astronómicas para simular

Para simular órbitas es cómodo medir las distancias en ua, el tiempo en años y las masas en masas solares. En esas unidades, la tercera ley de Kepler para la Tierra ($T = 1$, $a = 1$, $M = 1$) obliga a que $G = 4\pi^2$. Así evitamos números enormes y pequeños mezclados.

El método es el de Verlet de la semana 17, ahora con dos cuerpos que se atraen mutuamente: cada uno recibe la fuerza del otro, igual y opuesta.

In [ ]:
G = 4 * np.pi**2      # ua³ / (masa solar · año²)

def aceleraciones(pos, m):
    """pos: array 2×2 (una fila por cuerpo). Devuelve las aceleraciones de los dos."""
    d = pos[1] - pos[0]
    r3 = np.linalg.norm(d) ** 3
    a0 =  G * m[1] / r3 * d       # el cuerpo 0 es atraído hacia el 1
    a1 = -G * m[0] / r3 * d       # y el 1 hacia el 0
    return np.array([a0, a1])

def simular(m, pos, vel, t_total, dt=1e-3):
    pos, vel = pos.astype(float).copy(), vel.astype(float).copy()
    a = aceleraciones(pos, m)
    n = int(t_total / dt)
    hist = np.zeros((n + 1, 2, 2))
    hist[0] = pos
    for i in range(n):
        vel += 0.5 * dt * a
        pos += dt * vel
        a = aceleraciones(pos, m)
        vel += 0.5 * dt * a
        hist[i + 1] = pos
    return hist

def condiciones_iniciales(m1, m2, a=1.0, factor_v=1.0):
    """Separación a sobre el eje x, centro de masas en reposo en el origen.
    factor_v = 1 da órbita circular; menor que 1, elíptica."""
    M = m1 + m2
    v_rel = factor_v * np.sqrt(G * M / a)
    pos = np.array([[-m2 / M * a, 0.0], [m1 / M * a, 0.0]])
    vel = np.array([[0.0, -m2 / M * v_rel], [0.0, m1 / M * v_rel]])
    return pos, vel

## 3. Dos cuerpos con distintas proporciones de masa

Simulamos tres casos, siempre con 1 ua de separación media y órbitas algo elípticas: masas iguales, una el triple que la otra y una diez veces la otra.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (m1, m2) in zip(axs, [(1, 1), (3, 1), (10, 1)]):
    m = np.array([m1, m2], dtype=float)
    pos, vel = condiciones_iniciales(m1, m2, factor_v=0.8)
    h = simular(m, pos, vel, t_total=1.0)
    cm = (m[0] * h[:, 0] + m[1] * h[:, 1]) / m.sum()
    ax.plot(h[:, 0, 0], h[:, 0, 1], label=f"masa {m1}")
    ax.plot(h[:, 1, 0], h[:, 1, 1], label=f"masa {m2}")
    ax.plot(cm[:, 0], cm[:, 1], "k+", ms=12, label="centro de masas")
    for k in range(0, len(h), len(h) // 8):            # unas líneas que unen los dos cuerpos
        ax.plot(h[k, :, 0], h[k, :, 1], color="gray", lw=0.5)
    ax.set_aspect("equal"); ax.set_title(f"masas {m1} : {m2}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

Las líneas grises unen los dos cuerpos en distintos instantes: todas pasan por el centro de masas, que se queda quieto. Con masas iguales, las dos elipses son idénticas; con una masa diez veces mayor, el cuerpo pesado apenas se mueve.

## 4. El centro de masas no se acelera

Ahora le damos al sistema una velocidad inicial de conjunto: sumamos la misma velocidad $(0{,}5, 0{,}3)$ ua/año a los dos cuerpos. Cada uno hace un movimiento complicado, pero el centro de masas debe ir en línea recta a velocidad constante.

In [ ]:
m = np.array([3.0, 1.0])
pos, vel = condiciones_iniciales(3, 1, factor_v=0.9)
vel += np.array([0.5, 0.3])
h = simular(m, pos, vel, t_total=3.0)
cm = (m[0] * h[:, 0] + m[1] * h[:, 1]) / m.sum()

plt.figure(figsize=(8, 5))
plt.plot(h[:, 0, 0], h[:, 0, 1], label="masa 3")
plt.plot(h[:, 1, 0], h[:, 1, 1], label="masa 1")
plt.plot(cm[:, 0], cm[:, 1], "k--", label="centro de masas")
plt.gca().set_aspect("equal"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Dos cuerpos que se trasladan: el centro de masas va en línea recta")
plt.show()

## 5. El periodo y la tercera ley de Kepler

**Ejercicio.** Simula dos cuerpos de masas 1 y 1 (masas solares) con separación circular de 1 ua. Mide cuánto tardan en dar una vuelta: busca cuándo el vector que va del cuerpo 0 al 1 vuelve a apuntar hacia la derecha (el ángulo con `np.arctan2` pasa de negativo a positivo). Compara con $T = \sqrt{a^3/(m_1 + m_2)}$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
def periodo_simulado(m1, m2, dt=1e-4):
    m = np.array([m1, m2], dtype=float)
    pos, vel = condiciones_iniciales(m1, m2)
    h = simular(m, pos, vel, t_total=1.5, dt=dt)
    d = h[:, 1] - h[:, 0]
    ang = np.arctan2(d[:, 1], d[:, 0])
    cruces = np.where((ang[:-1] < 0) & (ang[1:] >= 0))[0]
    return cruces[0] * dt

for m1, m2 in [(1, 1e-6), (1, 0.001), (1, 1), (2, 1)]:
    T_sim = periodo_simulado(m1, m2)
    T_kep = np.sqrt(1.0 / (m1 + m2))
    print(f"masas {m1} y {m2}:  T simulado = {T_sim:.4f} años   T Kepler = {T_kep:.4f} años")

Con dos soles, el periodo es $1/\sqrt{2} \approx 0{,}71$ años: la masa total es el doble, y la órbita se recorre más deprisa.

## 6. El bamboleo del Sol

Simulamos el Sol y Júpiter durante dos órbitas de Júpiter, con datos reales redondeados, y dibujamos el movimiento del Sol en kilómetros alrededor del centro de masas, junto con el tamaño del propio Sol.

In [ ]:
m = np.array([1.0, 1 / 1047.6])                  # Sol y Júpiter, en masas solares
pos, vel = condiciones_iniciales(m[0], m[1], a=5.203)
h = simular(m, pos, vel, t_total=24, dt=2e-3)
cm = (m[0] * h[:, 0] + m[1] * h[:, 1]) / m.sum()
sol_km = (h[:, 0] - cm) * 1.496e8

plt.figure(figsize=(5.5, 5.5))
ang = np.linspace(0, 2 * np.pi, 200)
plt.fill(696_000 * np.cos(ang) + sol_km[0, 0], 696_000 * np.sin(ang) + sol_km[0, 1], color="orange", alpha=0.3, label="el Sol (posición inicial)")
plt.plot(sol_km[:, 0], sol_km[:, 1], label="trayectoria del centro del Sol")
plt.plot(0, 0, "k+", ms=12, label="centro de masas")
plt.gca().set_aspect("equal"); plt.legend(fontsize=8, loc="upper right"); plt.xlabel("km"); plt.ylabel("km")
plt.title("El Sol alrededor del centro de masas Sol-Júpiter")
plt.show()

v_sol = np.linalg.norm(np.diff(sol_km, axis=0), axis=1).mean() / (2e-3 * 365.25 * 86400)
print(f"Distancia del centro del Sol al centro de masas: {np.linalg.norm(sol_km[0]):,.0f} km   velocidad del Sol: {v_sol*1000:.1f} m/s")

El centro de masas queda justo fuera del borde del Sol, y el Sol se mueve a unos 12,5 m/s. Desde otra estrella, ese movimiento se detectaría como un pequeño vaivén en la velocidad del Sol, con un periodo de 11,9 años: así se descubren muchos exoplanetas.

## 7. Experimenta

1. Repite el bamboleo con Saturno (masa 1/3498 de la del Sol, a 9,54 ua). ¿Qué amplitud y qué velocidad produce?
2. (Más difícil) Pon tres cuerpos: Sol, Júpiter y Saturno, y dibuja el movimiento del Sol durante 60 años. Necesitas generalizar `aceleraciones` a $N$ cuerpos: para cada cuerpo, suma las atracciones de todos los demás.

In [ ]:
# Tu experimento aquí